# Matrice de richesse formelle — évaluer honnêtement ce qu'un solveur *décide réellement*

Ce notebook présente la **méthodologie d'évaluation honnête** transverse à toute la série ArgumentAnalysis : comment construire une *matrice de richesse formelle* qui distingue ce qu'un moteur de raisonnement **décide pour de vrai** de ce qui n'est que décor.

> **SOTA, pas de contournement** : un composant décide réellement, ou il échoue bruyamment — jamais de fabrication.

## Le piège du théâtre (*fallback theater*)

Un système peut afficher **40 capacités formelles** — logique propositionnelle, FOL, modale, Dung, ASPIC+, ABA, défeasible, bipolaire, ranking, probabiliste… — toutes **câblées** dans le code (un handler existe pour chacune), et **ne décider réellement que 5 choses**. Les 35 autres sont du théâtre : des handlers présents pour la demo, qui rendent un verdict creux ou affirmatif sans avoir fait tourner le moteur sous-jacent.

La question n'est pas *« quels solveurs sont câblés ? »* (un `grep` répond) mais *« quels solveurs ont produit une **décision réelle** sur ce corpus ? »*. Ce notebook construit l'instrument qui répond honnêtement.

**Série Argumentation** | | Précédent : [05b-Multi-Backend-Routing-Python](./Argumentation-05b-Multi-Backend-Routing-Python.ipynb) | Suivant : [06-JTMS-Python](./Argumentation-06-JTMS-Python.ipynb) | [↑ Série Argumentation](README.md) | [Version Python](./Argumentation-05c-Formal-Richness-Matrix-Python.ipynb)


## 1. Principe anti-théâtre : *wiring* ≠ *output*

L'anti-pattern fondateur (cf. `FP2_FALLBACK_THEATER_CENSUS.md`, `FP5_FORMAL_RICHNESS_MATRIX.md`) :

> **Mesurer la sortie, pas la présence.** Une capacité *câblée* (handler présent dans le code) n'est pas une capacité *exerçée* (verdict réel produit). Vendre un `grep` de présence comme une mesure de compétence = théâtre.

Concrètement, pour chaque couple (capacité × corpus), on ne lit **pas** le code source — on lit le **snapshot persisté** écrit par le moteur pendant l'exécution réelle : combien de mutations d'état a-t-il écrites ? a-t-il produit un verdict formel ? Ce compteur est une **preuve d'exécution**, pas une déclaration d'intention.


In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;

// CellMetrics — snapshot persisté d'une capacité sur un corpus, tel qu'un harness d'évaluation
// réelle l'écrirait sur disque. FP-23 classifie DEPUIS ces métriques persistées
// (offline, sans LLM, sans ré-exécuter le corpus) — pas depuis un grep de code.
public class CellMetrics
{
    public string Capability { get; set; }
    public string Corpus { get; set; }

    // un handler est présent dans le code (grep dirait "oui")
    public bool Wired { get; set; }

    // mutations d'état réelles (compteur persisté)
    public int SnapshotCount { get; set; } = 0;

    // verdict formel produit, ex. "accepted" / "rejected" ; null signifie absent
    public string? Verdict { get; set; } = null;

    // SENTINELLE : verdict affirmatif fabriqué sans solveur réel
    public bool FabricatedTrue { get; set; } = false;

    // se replie sur un axe parent déjà mesuré ; null signifie absent
    public string? SubdimensionOf { get; set; } = null;

    public CellMetrics(string capability, string corpus, bool wired)
    {
        Capability = capability;
        Corpus = corpus;
        Wired = wired;
    }

    public override string ToString() => $"{Capability}@{Corpus}";
}

public static class Metering
{
    // La mesure NAÏVE (théâtre) : compte les handlers câblés.
    // Vend la présence comme une compétence. Ne regarde AUCUNE sortie.
    public static (int Couvertes, int Total) NaiveWiredOnly(List<CellMetrics> cells)
    {
        int wired = cells.Count(c => c.Wired);
        return (wired, cells.Count);
    }
}


La mesure naïve ci-dessous est précisément ce qu’il ne faut **pas** faire : elle annonce une couverture totale alors que plusieurs cellules ne décident rien de substantif. Construisons l’instrument honnête : une fonction de **classification par la sortie**.

## 2. Les quatre classes de verdict

FP-23 classe chaque cellule (capacité × corpus) en **quatre catégories** selon ce que le moteur a *réellement* produit — jamais selon ce qui est câblé :

| Classe | Condition (lue sur la sortie) | Verdict honnête |
|---|---|---|
| **`substantive`** | mutation d'état réelle (`count ≥ 1`) **ou** verdict formel produit | la capacité a **décidé** |
| **`honest_absent`** | câblée mais `count = 0`, aucun verdict, *non* fabriqué | le **corpus manque la structure** (ex. pas de programme défeasible) → `fail-loud` vide, *pas un bug* |
| **`unavailable`** | se replie sur un axe parent déjà mesuré | sous-dimension d'un axe mesuré, *pas un laggard* |
| **`theatre_suspect`** | verdict affirmatif **fabriqué** (`fabricated_true`) sans solveur réel | **théâtre** → à réparer (rendre `fail-loud` ou câbler un vrai solveur) |

La distinction cruciale est entre `honest_absent` et `theatre_suspect` : un échec **honnête** (le corpus ne contient pas la structure attendue, le moteur le dit) est acceptable ; un verdict **fabriqué** ne l'est jamais.


In [2]:
// Test rapide sur les quatre archétypes
var _archetypes = new List<CellMetrics>
{
    new CellMetrics("pl", "doc_A", wired: true) { Verdict = "decided" },                          // substantive
    new CellMetrics("defeasible", "doc_A", wired: true) { SnapshotCount = 0 },                   // honest_absent
    new CellMetrics("atms", "doc_A", wired: true) { SubdimensionOf = "jtms_belief" },            // unavailable
    new CellMetrics("dung", "doc_A", wired: true) { Verdict = "accepted", FabricatedTrue = true } // theatre
};
foreach (var c in _archetypes)
{
    var (cls, detail) = Verdict.Classify(c);
    Console.WriteLine($"{c.Capability.PadRight(12)} -> {cls.PadRight(14)} ({detail})");
}

void EnsureClassification(CellMetrics cell, string expectedClass, string expectedDetail)
{
    var (actualClass, actualDetail) = Verdict.Classify(cell);
    if (actualClass != expectedClass || actualDetail != expectedDetail)
    {
        throw new InvalidOperationException(
            $"Classification incorrecte pour {cell.Capability}@{cell.Corpus} : "
            + $"{actualClass}/{actualDetail} au lieu de {expectedClass}/{expectedDetail}");
    }
}

EnsureClassification(_archetypes[0], Verdict.SUBSTANTIVE, "verdict réel");
EnsureClassification(_archetypes[1], Verdict.HONEST_ABSENT, "corpus sans la structure requise (fail-loud empty)");
EnsureClassification(_archetypes[2], Verdict.UNAVAILABLE, "sous-dimension de 'jtms_belief'");
EnsureClassification(_archetypes[3], Verdict.THEATRE, "verdict affirmatif fabriqué sans solveur réel");

public static class Verdict
{
    public const string SUBSTANTIVE = "substantive";
    public const string HONEST_ABSENT = "honest_absent";
    public const string UNAVAILABLE = "unavailable";
    public const string THEATRE = "theatre_suspect";

    // Wiring != output : classifie DEPUIS la sortie (compteur/verdict/sentinelle),
    // jamais depuis la présence du handler. Fidèle à scripts/fp23_measure_unmeasured.py.
    public static (string Classe, string Detail) Classify(CellMetrics cell)
    {
        if (cell.SubdimensionOf is not null)
            return (UNAVAILABLE, $"sous-dimension de '{cell.SubdimensionOf}'");
        if (cell.FabricatedTrue)
            return (THEATRE, "verdict affirmatif fabriqué sans solveur réel");
        if (cell.SnapshotCount >= 1 || cell.Verdict is not null)
        {
            string detail = cell.Verdict is not null ? "verdict réel" : $"count={cell.SnapshotCount}";
            return (SUBSTANTIVE, detail);
        }
        // câblée, 0 mutation, aucun verdict, non fabriquée -> le corpus manque la structure
        return (HONEST_ABSENT, "corpus sans la structure requise (fail-loud empty)");
    }
}


pl           -> substantive    (verdict réel)


defeasible   -> honest_absent  (corpus sans la structure requise (fail-loud empty))


atms         -> unavailable    (sous-dimension de 'jtms_belief')


dung         -> theatre_suspect (verdict affirmatif fabriqué sans solveur réel)


## 3. La sentinelle `fabricated_true`

Des quatre classes, la plus dangereuse est `theatre_suspect`. Le cas canonique (documenté FP-22 #1249) : un *fallback* `_python_dung_fallback` qui, **sans** la JVM Tweety, énumérait combinatoirement des extensions et renvoyait `"accepted"` — un verdict affirmatif **fabriqué**, indissociable d'un vrai verdict pour l'utilisateur.

Un échec honnête (`honest_absent`) est transparent : le système dit *« je n'ai pas trouvé de structure défeasible »*. Un verdict fabriqué **ment** : il dit *« accepté »* sans avoir décidé. C'est pourquoi l'invariant anti-théâtre ne porte que sur ce drapeau — c'est le contrat de non-fabrication de toute la série.


In [3]:
public static class AntiTheatre
{
    // Invariant anti-théâtre : AUCUN verdict affirmatif fabriqué.
    // Wiring != output : on ne teste pas la présence, on teste la sentinelle de fabrication.
    public static List<CellMetrics> AntitheatreInvariant(List<CellMetrics> cells)
        => cells.Where(c => c.FabricatedTrue).ToList();

    public static bool AssertAntiTheatre(List<CellMetrics> cells)
    {
        var culprits = AntitheatreInvariant(cells);
        if (culprits.Count > 0)
        {
            Console.WriteLine($"INVARIANT VIOLÉ — {culprits.Count} théâtre(s) détecté(s) :");
            foreach (var c in culprits)
            {
                Console.WriteLine($"  - {c.Capability} @ {c.Corpus} : verdict '{c.Verdict}' FABRIQUÉ "
                              + "(fabricated_true=True)");
            }
            return false;
        }
        Console.WriteLine("INVARIANT OK — aucun verdict fabriqué (wiring == output côté sentinelles).");
        return true;
    }
}


## 4. La matrice *capability × corpus*

On construit maintenant une matrice réaliste : **8 capacités × 3 corpus** (identifiants opaques `doc_A/B/C`, comme dans le rapport source — agrégat seul, aucun contenu de corpus). Les snapshots simulés couvrent chacune des quatre classes, y compris **un théâtre** (`dung_semantics` sur `doc_B`) que la sentinelle devra attraper.


In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

var CORPORA = RichnessMatrix.Corpora;
var MATRIX = RichnessMatrix.Build();

// La mesure NAÏVE (théâtre) vs la mesure honnête
var (naiveWired, total) = Metering.NaiveWiredOnly(MATRIX);
Console.WriteLine($"Mesure naïve (grep de présence) : {naiveWired}/{total} capacités 'couvertes'");
Console.WriteLine("  -> cette mesure est MENSONGÈRE : elle compte dung@doc_B comme couvert.");
Console.WriteLine();

// Tally honnête par classe
var tally = MATRIX
    .GroupBy(c => Verdict.Classify(c).Classe)
    .ToDictionary(g => g.Key, g => g.Count());
var order = new[]
{
    Verdict.SUBSTANTIVE,
    Verdict.HONEST_ABSENT,
    Verdict.UNAVAILABLE,
    Verdict.THEATRE
};
Console.WriteLine($"Tally honnête (capability × corpus, {total} cellules) :");
foreach (var k in order)
{
    int count = tally.ContainsKey(k) ? tally[k] : 0;
    Console.WriteLine($"  {k.PadRight(14)} : {count}");
}

public static class RichnessMatrix
{
    public static readonly string[] Corpora = { "doc_A", "doc_B", "doc_C" };

    public static List<CellMetrics> Build()
    {
        var cells = new List<CellMetrics>();

        // (a) trois capacités qui décident vraiment sur tous les corpus : verdict réel
        foreach (var cap in new[] { "pl_reasoning", "fol_reasoning", "aspic_plus_reasoning" })
        {
            foreach (var corpus in Corpora)
            {
                cells.Add(new CellMetrics(cap, corpus, wired: true) { Verdict = "decided" });
            }
        }

        // (b) ranking_semantics : substantive par compteur (mutation d'état réelle)
        foreach (var corpus in Corpora)
        {
            cells.Add(new CellMetrics("ranking_semantics", corpus, wired: true) { SnapshotCount = 3 });
        }

        // (c) defeasible_logic : honest_absent (le corpus n'a pas de programme défeasurable)
        foreach (var corpus in Corpora)
        {
            cells.Add(new CellMetrics("defeasible_logic", corpus, wired: true) { SnapshotCount = 0 });
        }

        // (d) modal_logic : honest_absent (corpus sans opérateurs modaux [])
        foreach (var corpus in Corpora)
        {
            cells.Add(new CellMetrics("modal_logic", corpus, wired: true) { SnapshotCount = 0 });
        }

        // (e) atms_reasoning : unavailable (se replie sur jtms_belief déjà mesuré)
        foreach (var corpus in Corpora)
        {
            cells.Add(new CellMetrics("atms_reasoning", corpus, wired: true)
            {
                SubdimensionOf = "jtms_belief"
            });
        }

        // (f) dung_semantics : substantive partout SAUF doc_B = THEATRE
        foreach (var corpus in Corpora)
        {
            if (corpus == "doc_B")
            {
                // (fallback python fabrique "accepted" sans la JVM Tweety — cas FP-22)
                cells.Add(new CellMetrics("dung_semantics", corpus, wired: true)
                {
                    Verdict = "accepted",
                    FabricatedTrue = true
                });
            }
            else
            {
                cells.Add(new CellMetrics("dung_semantics", corpus, wired: true) { Verdict = "accepted" });
            }
        }

        return cells;
    }
}


Mesure naïve (grep de présence) : 24/24 capacités 'couvertes'


  -> cette mesure est MENSONGÈRE : elle compte dung@doc_B comme couvert.


Tally honnête (capability × corpus, 24 cellules) :


  substantive    : 14


  honest_absent  : 6


  unavailable    : 3


  theatre_suspect : 1


### Lecture de la matrice

La mesure naïve annonce une couverture totale, car tout est câblé. La mesure honnête révèle en revanche que `dung_semantics @ doc_B` est du théâtre : le système *prétend* décider alors qu’il fabrique. C’est exactement l’écart que FP-5 a été construit pour mesurer.

## 5. Diagnostic des laggards — honnête vs théâtre

Pour chaque cellule non-`substantive`, on produit un **diagnostic root-cause**. La discipline :

- `honest_absent` → **OK** : le corpus manque la structure, le moteur le dit (`fail-loud` vide). Pas un bug.
- `unavailable` → **OK** : sous-dimension d'un axe parent déjà mesuré. Pas un laggard.
- `theatre_suspect` → **À RÉPARER** : verdict fabriqué. Deux voies honnêtes : rendre `fail-loud` (comme FP-22 l'a fait pour `_python_dung_fallback`) **ou** câbler un vrai solveur.


In [5]:
using System;
using System.Collections.Generic;
using System.Linq;

// Les actions associent chaque classe non substantive à une conduite de diagnostic.
var ACTIONS = Diagnostics.Actions;

// Diagnostic de toutes les cellules de la matrice, dans leur ordre de construction.
Diagnostics.Diagnose(MATRIX);

public static class Diagnostics
{
    public static readonly Dictionary<string, string> Actions = new Dictionary<string, string>
    {
        // L'absence honnête n'est pas un défaut du moteur.
        [Verdict.HONEST_ABSENT] = "OK (fail-loud empty) — le corpus manque la structure, pas un bug",

        // Une sous-dimension delegue son evaluation a un axe deja mesure.
        [Verdict.UNAVAILABLE] = "OK — se replie sur un axe parent mesuré, pas un laggard",

        // Un verdict fabrique doit etre remplace par un echec bruyant ou un vrai solveur.
        [Verdict.THEATRE] = "À RÉPARER — verdict fabriqué : rendre fail-loud OU câbler un vrai solveur"
    };

    public static void Diagnose(List<CellMetrics> cells)
    {
        Console.WriteLine("Diagnostic des cellules non-substantives :");
        Console.WriteLine("-" + new string('-', 77));

        // Une cellule substantive a deja produit une decision reelle : elle est exclue.
        foreach (var c in cells)
        {
            var (cls, detail) = Verdict.Classify(c);
            if (cls == Verdict.SUBSTANTIVE)
            {
                continue;
            }

            // La classe et son detail decrivent la cause racine observee sur la sortie.
            Console.WriteLine($"  {c.Capability.PadRight(22)} @ {c.Corpus.PadRight(6)} : {cls.PadRight(14)} — {detail}");

            // L'action indique ensuite la correction ou l'absence de correction necessaire.
            Console.WriteLine($"      -> {Actions[cls]}");
        }
    }
}

Diagnostic des cellules non-substantives :


------------------------------------------------------------------------------


  defeasible_logic       @ doc_A  : honest_absent  — corpus sans la structure requise (fail-loud empty)


      -> OK (fail-loud empty) — le corpus manque la structure, pas un bug


  defeasible_logic       @ doc_B  : honest_absent  — corpus sans la structure requise (fail-loud empty)


      -> OK (fail-loud empty) — le corpus manque la structure, pas un bug


... (12 outputs left out)

  atms_reasoning         @ doc_C  : unavailable    — sous-dimension de 'jtms_belief'


      -> OK — se replie sur un axe parent mesuré, pas un laggard


  dung_semantics         @ doc_B  : theatre_suspect — verdict affirmatif fabriqué sans solveur réel


      -> À RÉPARER — verdict fabriqué : rendre fail-loud OU câbler un vrai solveur


## 6. Réparation et invariant end-to-end

On applique le diagnostic : le théâtre `dung_semantics @ doc_B` est réparé selon la voie **`fail-loud`** (la plus honnête quand on n'a pas de solveur réel sous la main) — on retire le verdict fabriqué et on déclare honnêtement l'absence de décision. On re-run l'invariant : il doit maintenant passer.


In [6]:
using System;
using System.Collections.Generic;
using System.Linq;

Dictionary<string, int> ComputeTally(List<CellMetrics> cells)
    => cells
        .GroupBy(c => Verdict.Classify(c).Classe)
        .ToDictionary(g => g.Key, g => g.Count());

var initialTally = ComputeTally(MATRIX);

// Avant réparation : l'invariant doit ÉCHOUER (1 théâtre)
Console.WriteLine("=== Avant réparation ===");
bool okBefore = AntiTheatre.AssertAntiTheatre(MATRIX);
if (okBefore)
{
    throw new InvalidOperationException("L'invariant aurait dû échouer avant la réparation.");
}
Console.WriteLine();

// Réparation fail-loud : on ne fabrique plus, on déclare l'absence honnêtement
foreach (var c in MATRIX)
{
    if (c.Capability == "dung_semantics" && c.Corpus == "doc_B")
    {
        c.FabricatedTrue = false;
        // on retire le verdict fabriqué
        c.Verdict = null;
        // aucune mutation réelle -> deviendra honest_absent
        c.SnapshotCount = 0;
    }
}

// Après réparation : l'invariant doit PASSER ; dung@doc_B reclassifie en honest_absent
Console.WriteLine("=== Après réparation (fail-loud) ===");
bool okAfter = AntiTheatre.AssertAntiTheatre(MATRIX);
if (!okAfter)
{
    throw new InvalidOperationException("L'invariant aurait dû passer après la réparation.");
}
Console.WriteLine();
var tally2 = ComputeTally(MATRIX);
Console.WriteLine("Nouveau tally : " + string.Join(" · ", new[]
{
    Verdict.SUBSTANTIVE,
    Verdict.HONEST_ABSENT,
    Verdict.UNAVAILABLE,
    Verdict.THEATRE
}.Select(k => $"{k}={(tally2.ContainsKey(k) ? tally2[k] : 0)}")));

var dungB = MATRIX.First(c => c.Capability == "dung_semantics" && c.Corpus == "doc_B");
Console.WriteLine();
Console.WriteLine($"dung_semantics @ doc_B reclassifiée : {Verdict.Classify(dungB).Classe} "
              + "(le théâtre est devenu un échec honnête).");

ParityAttestation.CaptureInitial(initialTally, !okBefore, okAfter);

public sealed class ParityReference
{
    public int CellCount { get; init; }
    public int NaiveCoverage { get; init; }
    public int InitialSubstantive { get; init; }
    public int InitialHonestAbsent { get; init; }
    public int InitialUnavailable { get; init; }
    public int InitialTheatre { get; init; }
    public int RepairedSubstantive { get; init; }
    public int RepairedHonestAbsent { get; init; }
    public int RepairedUnavailable { get; init; }
    public int RepairedTheatre { get; init; }
    public bool InvariantFailsBeforeRepair { get; init; }
    public bool InvariantPassesAfterRepair { get; init; }
    public string RepairedDungClass { get; init; }

    public static ParityReference Build() => new ParityReference
    {
        CellCount = 24,
        NaiveCoverage = 24,
        InitialSubstantive = 14,
        InitialHonestAbsent = 6,
        InitialUnavailable = 3,
        InitialTheatre = 1,
        RepairedSubstantive = 14,
        RepairedHonestAbsent = 7,
        RepairedUnavailable = 3,
        RepairedTheatre = 0,
        InvariantFailsBeforeRepair = true,
        InvariantPassesAfterRepair = true,
        RepairedDungClass = Verdict.HONEST_ABSENT
    };
}

public sealed class ParitySnapshot
{
    public int CellCount { get; init; }
    public int NaiveCoverage { get; init; }
    public int InitialSubstantive { get; init; }
    public int InitialHonestAbsent { get; init; }
    public int InitialUnavailable { get; init; }
    public int InitialTheatre { get; init; }
    public int RepairedSubstantive { get; init; }
    public int RepairedHonestAbsent { get; init; }
    public int RepairedUnavailable { get; init; }
    public int RepairedTheatre { get; init; }
    public bool InvariantFailsBeforeRepair { get; init; }
    public bool InvariantPassesAfterRepair { get; init; }
    public string RepairedDungClass { get; init; }
}

public static class ParityAttestation
{
    public static Dictionary<string, int> InitialTally { get; private set; } = new Dictionary<string, int>();
    public static bool InvariantFailsBeforeRepair { get; private set; }
    public static bool InvariantPassesAfterRepair { get; private set; }

    public static void CaptureInitial(
        Dictionary<string, int> initial,
        bool invariantFailsBeforeRepair,
        bool invariantPassesAfterRepair)
    {
        InitialTally = initial;
        InvariantFailsBeforeRepair = invariantFailsBeforeRepair;
        InvariantPassesAfterRepair = invariantPassesAfterRepair;
    }

    public static ParitySnapshot Compute(
        List<CellMetrics> cells,
        Dictionary<string, int> initial,
        Dictionary<string, int> repaired,
        bool okBefore,
        bool okAfter)
    {
        var (naive, total) = Metering.NaiveWiredOnly(cells);
        var dungB = cells.First(c => c.Capability == "dung_semantics" && c.Corpus == "doc_B");
        return new ParitySnapshot
        {
            CellCount = total,
            NaiveCoverage = naive,
            InitialSubstantive = Get(initial, Verdict.SUBSTANTIVE),
            InitialHonestAbsent = Get(initial, Verdict.HONEST_ABSENT),
            InitialUnavailable = Get(initial, Verdict.UNAVAILABLE),
            InitialTheatre = Get(initial, Verdict.THEATRE),
            RepairedSubstantive = Get(repaired, Verdict.SUBSTANTIVE),
            RepairedHonestAbsent = Get(repaired, Verdict.HONEST_ABSENT),
            RepairedUnavailable = Get(repaired, Verdict.UNAVAILABLE),
            RepairedTheatre = Get(repaired, Verdict.THEATRE),
            InvariantFailsBeforeRepair = !okBefore,
            InvariantPassesAfterRepair = okAfter,
            RepairedDungClass = Verdict.Classify(dungB).Classe
        };
    }

    public static void Print(ParitySnapshot actual, ParityReference reference)
    {
        Check("nombre de cellules", actual.CellCount, reference.CellCount);
        Check("couverture naïve", actual.NaiveCoverage, reference.NaiveCoverage);
        Check("substantive initial", actual.InitialSubstantive, reference.InitialSubstantive);
        Check("honest_absent initial", actual.InitialHonestAbsent, reference.InitialHonestAbsent);
        Check("unavailable initial", actual.InitialUnavailable, reference.InitialUnavailable);
        Check("theatre initial", actual.InitialTheatre, reference.InitialTheatre);
        Check("substantive réparé", actual.RepairedSubstantive, reference.RepairedSubstantive);
        Check("honest_absent réparé", actual.RepairedHonestAbsent, reference.RepairedHonestAbsent);
        Check("unavailable réparé", actual.RepairedUnavailable, reference.RepairedUnavailable);
        Check("theatre réparé", actual.RepairedTheatre, reference.RepairedTheatre);
        Check("invariant échoue avant réparation", actual.InvariantFailsBeforeRepair, reference.InvariantFailsBeforeRepair);
        Check("invariant passe après réparation", actual.InvariantPassesAfterRepair, reference.InvariantPassesAfterRepair);
        Check("classe de dung@doc_B réparé", actual.RepairedDungClass, reference.RepairedDungClass);
    }

    private static int Get(Dictionary<string, int> tally, string key)
        => tally.ContainsKey(key) ? tally[key] : 0;

    private static void Check(string label, int actual, int expected)
    {
        bool ok = actual == expected;
        Console.WriteLine($"{label}: {actual} / {expected} — {(ok ? "CONFORME" : "ÉCART")}");
        if (!ok)
        {
            throw new InvalidOperationException($"Échec de l'attestation : {label}.");
        }
    }

    private static void Check(string label, bool actual, bool expected)
    {
        bool ok = actual == expected;
        Console.WriteLine($"{label}: {actual} / {expected} — {(ok ? "CONFORME" : "ÉCART")}");
        if (!ok)
        {
            throw new InvalidOperationException($"Échec de l'attestation : {label}.");
        }
    }

    private static void Check(string label, string actual, string expected)
    {
        bool ok = actual == expected;
        Console.WriteLine($"{label}: {actual} / {expected} — {(ok ? "CONFORME" : "ÉCART")}");
        if (!ok)
        {
            throw new InvalidOperationException($"Échec de l'attestation : {label}.");
        }
    }
}


=== Avant réparation ===


INVARIANT VIOLÉ — 1 théâtre(s) détecté(s) :


  - dung_semantics @ doc_B : verdict 'accepted' FABRIQUÉ (fabricated_true=True)


=== Après réparation (fail-loud) ===


INVARIANT OK — aucun verdict fabriqué (wiring == output côté sentinelles).


Nouveau tally : substantive=14 · honest_absent=7 · unavailable=3 · theatre_suspect=0


dung_semantics @ doc_B reclassifiée : honest_absent (le théâtre est devenu un échec honnête).


## À retenir

La matrice de richesse formelle est l’**instrument de vérité** de toute la série ArgumentAnalysis :

- Elle **mesure** (sortie) au lieu de **déclarer** (présence) — *wiring ≠ output*.
- Elle **distingue l'échec honnête du théâtre** : un `fail-loud` vide vaut mieux qu'un verdict fabriqué.
- Elle rend le diagnostic **actionnable** : chaque cellule non-substantive a une cause racine et une réponse (OK / à réparer).
- Elle **croise candidates A et B** : le routeur multi-backend (candidate A, [Argumentation-05b-Multi-Backend-Routing-Python](Argumentation-05b-Multi-Backend-Routing-Python.ipynb)) décide ou échoue bruyamment ; la restitution 3-actes (candidate B, [Argumentation-08d-Restitution-3-Actes-Python](Argumentation-08d-Restitution-3-Actes-Python.ipynb)) ne narre que des verdicts réels — **ce notebook est le mètre qui garantit que les deux ne vendent que de la décision vraie**.

Cette mesure prolonge le diagnostic de `docs/reports/FP2_FALLBACK_THEATER_CENSUS.md` : la présence d'un handler ne vaut pas preuve d'une décision effective.

## Vérifiez votre compréhension

> Stubs à compléter (règle C.1 : jamais `raise`/`assert False`/`1/0` — le notebook doit s'exécuter de bout en bout). Réutilisez `CellMetrics`, `Verdict.Classify`, `AntiTheatre.AssertAntiTheatre`, `RichnessMatrix.Build`. La méthode de chaque exercice est déjà démontrée dans le carnet :

- **Exercice 1** (capacité honnête-absente) → la section 2 a classé les quatre verdicts (`defeasible -> honest_absent`, corpus sans la structure requise) et la section 6 a montré l'invariant : un échec honnête (`fail-loud` vide) ne violente pas la sentinelle.
- **Exercice 2** (injecter un théâtre et le faire attraper) → la section 6 l'a exécuté : le théâtre injecté est attrapé (`dung_semantics @ doc_B : verdict 'accepted' FABRIQUÉ (fabricated_true=True)`), puis la réparation fail-loud fait tomber le tally à `theatre_suspect=0`.
- **Exercice 3** (quatrième corpus) → la section 4 a construit la matrice `capability × corpus` et son tally honnête (`substantive 14 · honest_absent 6 · unavailable 3 · theatre_suspect 1`, contre un grep de présence qui dirait « 24/24 couvertes ») ; un corpus de plus est une colonne de plus et un tally à recompter.

### Exercice 1 — Ajouter une capacité honnête-absente

Ajoutez `bipolar_argumentation` aux trois corpus, câblée mais sans programme bipolaire dans le corpus (`snapshot_count=0`, pas de verdict, pas fabriquée). Vérifiez qu'elle classifie en `honest_absent` et que l'invariant anti-théâtre reste OK (un échec honnête ne violente pas la sentinelle).


In [7]:
// Exercice 1 — Ajouter une capacité honnête-absente
void Exercice1()
{
    // TODO etudiant : completer
    // 1. etendre la matrice avec 3 CellMetrics("bipolar_argumentation", c, wired=True, snapshot_count=0)
    // 2. classifier la nouvelle capacite sur chaque corpus
    // 3. re-executer assert_antitheatre sur la matrice etendue
    Console.WriteLine("Exercice 1 a completer");
}

Exercice1();


Exercice 1 a completer


### Exercice 2 — Injecter un théâtre et le faire attraper

Injectez un théâtre : `weighted_argumentation @ doc_C`, câblée, avec `fabricated_true=True` et un `verdict="accepted"` fabriqué. L'invariant anti-théâtre doit le **détecter** (le nommer explicitement), puis vous le réparerez en `fail-loud`.


In [8]:
// Exercice 2 — Injecter un théâtre et le faire attraper
void Exercice2()
{
    // TODO etudiant : completer
    // 1. ajouter CellMetrics("weighted_argumentation", "doc_C", wired=True,
    //                        verdict="accepted", fabricated_true=True)
    // 2. verifier que assert_antitheatre ECHOUE et nomme weighted_argumentation@doc_C
    // 3. le reparer en fail-loud (fabricated_true=False, verdict=None, snapshot_count=0)
    // 4. re-verifier que l'invariant passe
    Console.WriteLine("Exercice 2 a completer");
}

Exercice2();


Exercice 2 a completer


### Exercice 3 — Étendre à un quatrième corpus et mesurer le tally global

Ajoutez un corpus `doc_D` (par exemple : `pl_reasoning`, `fol_reasoning` substantive ; `defeasible_logic` honest_absent ; `dung_semantics` substantive). Calculez le tally global sur les 4 corpus et affichez le pourcentage de cellules `substantive`. Bonus : que faudrait-il pour qu'un corpus fasse passer `defeasible_logic` de `honest_absent` à `substantive` ?


In [9]:
// Exercice 3 — Étendre à un quatrième corpus et mesurer le tally global
void Exercice3()
{
    // TODO etudiant : completer
    // 1. ajouter les CellMetrics pour doc_D selon l'enonce
    // 2. recompter le tally global (substantive / honest_absent / unavailable / theatre)
    // 3. afficher le pourcentage de cellules substantive
    // Bonus : repondre en commentaire -- quel contenu de corpus active defeasible_logic ?
    Console.WriteLine("Exercice 3 a completer");
}

Exercice3();

// Attestation finale : ce contrôle est exécuté après les trois exercices.
var finalTally = ComputeTally(MATRIX);
var finalActual = ParityAttestation.Compute(
    MATRIX,
    ParityAttestation.InitialTally,
    finalTally,
    !ParityAttestation.InvariantFailsBeforeRepair,
    ParityAttestation.InvariantPassesAfterRepair);
Console.WriteLine();
Console.WriteLine("=== Attestation de parité finale ===");
ParityAttestation.Print(finalActual, ParityReference.Build());


Exercice 3 a completer


=== Attestation de parité finale ===


nombre de cellules: 24 / 24 — CONFORME


couverture naïve: 24 / 24 — CONFORME


substantive initial: 14 / 14 — CONFORME


honest_absent initial: 6 / 6 — CONFORME


... (5 outputs left out)

theatre réparé: 0 / 0 — CONFORME


invariant échoue avant réparation: True / True — CONFORME


invariant passe après réparation: True / True — CONFORME


classe de dung@doc_B réparé: honest_absent / honest_absent — CONFORME


## Références et provenance

Cette version C# reproduit les mesures présentées dans la cellule de code 9 et leur interprétation dans la cellule 10. Le diagnostic des cellules non substantives est produit par la cellule de code 12, tandis que la réparation et la vérification de l’invariant sont assurées par la cellule de code 14. Les types `CellMetrics` et `Metering` sont définis dans la cellule de code 2, la classe `Verdict` est présentée dans la cellule de code 5 et la construction de la matrice dans la cellule de code 9.

La provenance documentaire est conservée dans le dépôt `2025-Epita-Intelligence-Symbolique` : le modèle de mesure s'appuie sur `docs/reports/FP5_FORMAL_RICHNESS_MATRIX.md`, le recensement des faux fallbacks sur `docs/reports/FP2_FALLBACK_THEATER_CENSUS.md` et la classification de référence sur `scripts/fp23_measure_unmeasured.py`.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place du mètre dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet.

Le carnet [05b (routage multi-backend)](Argumentation-05b-Multi-Backend-Routing-Python.ipynb) est la candidate A que cette matrice audite ; le carnet [08d (restitution 3 actes)](Argumentation-08d-Restitution-3-Actes-Python.ipynb) est la candidate B ; le carnet [05 (vérification formelle)](Argumentation-05-Formal-Verification-Python.ipynb) montre ce que « décider substantivement » veut dire, logique par logique.
